In [ ]:
import pandas as pd
import numpy as np

In [12]:
import re

STATE_ZIP3_RANGES = {
    "Alabama": ((350, 369),), "Alaska": ((995, 999),), "Arizona": ((850, 865),),
    "Arkansas": ((716, 729),), "California": ((900, 961),), "Colorado": ((800, 816),),
    "Connecticut": ((60, 69),), "Delaware": ((197, 199),), "District of Columbia": ((200, 200),),
    "Florida": ((320, 349),), "Georgia": ((300, 319), (398, 398)), "Hawaii": ((967, 968),),
    "Idaho": ((832, 838),), "Illinois": ((600, 629),), "Indiana": ((460, 479),),
    "Iowa": ((500, 528),), "Kansas": ((660, 679),), "Kentucky": ((400, 427),),
    "Louisiana": ((700, 714),), "Maine": ((39, 49),), "Maryland": ((206, 219),),
    "Massachusetts": ((10, 27),), "Michigan": ((480, 499),), "Minnesota": ((550, 567),),
    "Mississippi": ((386, 397),), "Missouri": ((630, 658),), "Montana": ((590, 599),),
    "Nebraska": ((680, 693),), "Nevada": ((889, 898),), "New Hampshire": ((30, 38),),
    "New Jersey": ((70, 89),), "New Mexico": ((870, 884),), "New York": ((100, 149),),
    "North Carolina": ((270, 289),), "North Dakota": ((580, 588),), "Ohio": ((430, 459),),
    "Oklahoma": ((730, 731), (734, 749)), "Oregon": ((970, 979),),
    "Pennsylvania": ((150, 196),), "Rhode Island": ((28, 29),),
    "South Carolina": ((290, 299),), "South Dakota": ((570, 577),),
    "Tennessee": ((370, 385),), "Texas": ((733, 733), (750, 799), (885, 885)),
    "Utah": ((840, 847),), "Vermont": ((50, 59),),
    "Virginia": ((201, 201), (220, 246)), "Washington": ((980, 994),),
    "West Virginia": ((247, 268),), "Wisconsin": ((530, 549),), "Wyoming": ((820, 831),),
}
STATE_PATTERN = re.compile(
    r"\b(?P<state>" + "|".join(
        re.escape(state) for state in sorted(STATE_ZIP3_RANGES, key=len, reverse=True)
    ) + r")\b",
    re.IGNORECASE,
)
US_SUFFIX = re.compile(
    r"\bUnited States(?: of America)?\b(?:\s*\([^)]*\))?\s*$",
    re.IGNORECASE,
)
ZIP_PATTERN = re.compile(r"\b(?P<zip>\d{5})(?:-\d{4})?\b")
STATE_ZIP_ABBREVIATION_PATTERN = re.compile(
    r",\s*(?P<abbreviation>[A-Z]{2})\s+(?=\d{5}(?:-\d{4})?\b)"
)
STATE_ABBREVIATIONS = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "DC": "District of Columbia", "FL": "Florida", "GA": "Georgia", "HI": "Hawaii",
    "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming",
}


def get_state(address):
    state_match = STATE_PATTERN.search(address)
    if state_match:
        return state_match.group("state").title()

    abbreviation_match = STATE_ZIP_ABBREVIATION_PATTERN.search(address)
    if abbreviation_match:
        state = STATE_ABBREVIATIONS.get(abbreviation_match.group("abbreviation"))
        if state:
            return state

    if US_SUFFIX.search(address):
        zip_match = ZIP_PATTERN.search(address)
        if zip_match:
            zip3 = int(zip_match.group("zip")[:3])
            for state, ranges in STATE_ZIP3_RANGES.items():
                if any(start <= zip3 <= end for start, end in ranges):
                    return state

    return pd.NA


df = pd.read_csv("filtered.csv", low_memory=False)
df = df.dropna(subset=["address"]).copy()
df["state"] = df["address"].map(get_state)

print(f"Rows retained after removing null addresses: {len(df):,}")
print(f"Addresses with no identified U.S. state: {df['state'].isna().sum():,}")

ACTIVITY_PATTERNS = {
    "restaurant_dining": r"\b(?:restaurant|cafe|coffee shop|bakery|diner|bistro|eatery|food)\b",
    "skiing": r"\bski(?:ing|s)?\b|\bski (?:area|resort)\b",
    "cooking_experiences": r"\b(?:cooking|culinary) (?:class|experience|workshop)\b",
    "live_music": r"\b(?:live music|concert|music venue|theatre|theater|performing arts)\b",
    "nightlife": r"\b(?:bar|pub|night ?club|nightlife|cocktail lounge|bars & clubs)\b",
    "wellness": r"\b(?:spa|wellness|yoga|massage)\b",
    "museum_visiting": r"\bmuseum\b",
    "photography": r"\b(?:photography|photograph|photo studio)\b",
    "cycling": r"\b(?:cycling|bicycle|bike rental|bicycle rental)\b",
    "shopping": r"\b(?:shopping|shop|store|mall|retail)\b",
    "sightseeing": r"\b(?:tourist attraction|tourist_attraction|landmark|point_of_interest|general tourism|sightseeing|observation deck|viewpoint|botanical garden|park)\b",
    "adventure_sports": r"\b(?:adventure sports?|sports and recreation|climbing|rafting|zip ?line|paragliding|skydiving|bungee jumping|mountain)\b",
    "camping": r"\b(?:camping|campground|campsite|camp site)\b",
    "hiking": r"\b(?:hiking|hike|hiking trails?|mountain)\b",
    "running": r"\b(?:running|run club|running trails?)\b",
    "swimming": r"\b(?:swimming|swimming pool|water park)\b",
    "walking": r"\b(?:walking|walkway|walking tour|walking trails?|park)\b",
    "water_sports": r"\b(?:water sports?|beach|lake|boating|boat tours?|kayaking|canoeing|surfing|snorkeling|scuba diving|paddle ?boarding|jet skiing)\b",
    "wildlife_viewing": r"\b(?:wildlife|zoo|aquarium|safari|animal sanctuary|nature reserve|bird watching)\b",
    "festivals": r"\b(?:festival|festivals)\b",
    "food_tours": r"\b(?:food tours?|culinary tours?|food tastings?|wine tours?)\b",
    "historical_sites": r"\b(?:historic|historical|castle|palace|monument|memorial|religious sites?)\b",
}

ACTIVITY_PATTERNS = {
    activity: re.compile(pattern, re.IGNORECASE)
    for activity, pattern in ACTIVITY_PATTERNS.items()
}


def get_activities(category):
    if pd.isna(category):
        return []
    category_text = str(category).replace("_", " ")
    return [
        activity
        for activity, pattern in ACTIVITY_PATTERNS.items()
        if pattern.search(category_text)
    ]


df["activity"] = df["category"].map(get_activities)
print(f"Rows without a category match: {df['activity'].map(len).eq(0).sum():,}")

Rows retained after removing null addresses: 18,168
Addresses with no identified U.S. state: 2,931
Rows without a category match: 12,080


In [13]:
df.head

<bound method NDFrame.head of                                  name    latitude   longitude  \
0                     Top of The Rock  40.7593495  -73.979409   
1                   Statue of Liberty  40.6892494  -74.044500   
2                 The Channel Gardens  40.7585013  -73.978087   
3      The Metropolitan Museum of Art  40.7794366  -73.963244   
4              9/11 Memorial & Museum  40.7115776  -74.013336   
...                               ...         ...         ...   
26380                     Mai Tai Bar   21.276825 -157.828860   
26381               Round Table Pizza   21.282974 -157.837400   
26382                     Asahi Grill   21.297852 -157.853800   
26383         Waikiki Brewing Company    21.28584 -157.833540   
26384                    Moku Kitchen   21.298052 -157.861700   

                                                 address  \
0      Rockefeller Plaza, 10112, New York, United States   
1                       , 10004, New York, United States   
2        

In [17]:
df.to_csv("activities_updated.csv", index=False)